# 양자화

양자화(Quantization)는 모델의 파라미터(가중치)를 원래보다 훨씬 ㅈ거은 비트 수로 표현해서, 메모리 사용량과 연산량을 줄이는 기법입니다.

## 기본개념
일반적으로 딥러닝 모델의 가중치는 32비트(float32)또는 16비트(float16/bfloat16) 부동소수점으로 저장됩니다. 양자화는 이 값들을 4비트, 8비트 같은 훨씬 작은 정수 표현으로 압축하는 것입니다.

예를 들어 70억(7B) 파라미터 모델을 생각해보면 :
* float32 : 파라미터당 4바이트 -> 약 28GB
* float16 : 파라미터당 2바이트 -> 약 14GB
* 4비트 양자화 : 파라미터당 0.5바이트 -> 약 3.5GB

같은 모델인데도 메모리를 훨씬 적게 차지해서, 일반 GPU에서도 큰 모델을 돌릴 수 있게 됩니다.

## 트레이드오프

비트 수를 줄이면 정밀도가 떨어지기 때문에 모델 성능(정확도)이 약간 저하될 수 있습니다. 하지만 4비트/8비트 양자화는 성능 손실을 최소화하면서도 메모리를 크게 절약하도록 설계되어 있어서, 실무에서는 "거의 원본과 비슷한 품질 + 훨씬 적은 자원"이라는 실용적인 절충안으로 널리 쓰입니다.

In [1]:
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\airtr\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [2]:
# from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch
import gc
import os
from dotenv import load_dotenv


hf_token = os.getenv('HF_TOKEN')
if hf_token and hf_token.startswith("hf_"):
  print("HF key looks good so far")
else:
  print("HF key is not set - please click the key in the left sidebar")
login(hf_token, add_to_git_credential=True)

HF key looks good so far


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [7]:
LLAMA = "meta-llama/Llama-3.1-8B-Instruct"

PHI = "microsoft/Phi-4-mini-instruct"
GEMMA = "google/gemma-3-270m-it"
QWEN = "Qwen/Qwen3-4B-Instruct-2507"
DEEPSEEK = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

In [4]:
messages = [
    {"role": "user", "content": "Tell a joke for a room of Data Scientists"}
  ]

In [5]:
# Quantization Config - this allows us to load the model into memory and use less memory

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

In [8]:
# Tokenizer

tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

d:\STUDY\llm-core\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\airtr\.cache\huggingface\hub\models--meta-llama--Llama-3.1-8B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

AssertionError: Torch not compiled with CUDA enabled

In [ ]:
inputs

In [ ]:
# The model

model = AutoModelForCausalLM.from_pretrained(LLAMA, device_map="auto", quantization_config=quant_config)

In [ ]:
memory = model.get_memory_footprint() / 1e6
print(f"Memory footprint: {memory:,.1f} MB")

Transformer 모델 내부 들여다보기
다음 셀에서는 Llama의 HuggingFace `model` 객체를 출력합니다.
이 모델 객체는 신경망(Neural Network)이며, Python 프레임워크인 PyTorch로 구현되어 있습니다. 이 신경망은 2017년 Google 연구진이 고안한 아키텍처인 Transformer 아키텍처를 사용합니다.
이론을 깊이 다루지는 않겠지만, 이번 기회에 Transformer가 실제로 무엇인지에 대한 직관을 얻어볼 수 있습니다.
신경망을 완전히 처음 접하신다면, 기초를 다지기 위해 제 YouTube 입문 재생목록을 참고해 보세요.
이제 다음 셀에서 출력되는 신경망의 레이어들을 살펴보겠습니다. 다음 사항을 눈여겨보세요:

* 여러 개의 레이어로 구성되어 있습니다
* "embedding(임베딩)"이라는 것이 있는데, 이는 토큰을 4,096차원 벡터로 변환합니다. 이에 대해서는 5주차에 더 자세히 배울 예정입니다.
* 그 다음으로 "Decoder layer(디코더 레이어)"라고 불리는 레이어 그룹이 16세트(Llama 3.1의 경우 32세트) 있습니다. 각 디코더 레이어는 세 가지 종류의 레이어를 포함합니다: (a) self-attention(셀프 어텐션) 레이어 (b) multi-layer perceptron(MLP) 레이어 (c) batch norm(배치 정규화) 레이어.
* 마지막에는 LM Head 레이어가 있으며, 이 레이어가 최종 출력을 만들어냅니다

모델이 4비트로 양자화되었다는 언급도 확인해 보세요.
지금 단계에서 이론을 더 깊이 파고들 필요는 없지만, 원하신다면 제가 저희 둘의 지인에게 부탁해서 이 출력 결과를 가지고 각 레이어를 하나씩 살펴보는 튜토리얼을 만들어 달라고 했습니다. 이 튜토리얼에서는 각 지점에서의 차원(dimension)도 함께 살펴봅니다. 관심이 있으시다면, 다음 셀을 실행한 후에 이 튜토리얼을 진행해 보세요:

In [ ]:
# Execute this cell and look at what gets printed; investigate the layers

model

Transformer를 더 깊이 파고들고 싶다면
모델의 각 레이어를 살펴보는 것 외에도, PyTorch로 Llama를 구현한 실제 HuggingFace 코드를 직접 들여다볼 수도 있습니다.
다음은 HuggingFace Transformers 저장소입니다:
https://github.com/huggingface/transformers
그리고 그 안에서, Llama 4의 코드는 다음과 같습니다:
https://github.com/huggingface/transformers/blob/main/src/transformers/models/llama4/modeling_llama4.py
물론 이런 세부 내용까지 파고들 필요는 전혀 없습니다 - AI 엔지니어의 역할은 PyTorch로 Transformer를 직접 코딩하는 것이 아니라, LLM을 선택하고, 최적화하고, 파인튜닝하고, 적용하는 것이기 때문입니다. OpenAI, Meta를 비롯한 최전선의 연구소들은 이러한 모델을 만들고 학습시키는 데 수백만 달러를 투자했습니다. 하지만 관심이 있으시다면 한번 빠져볼 만한 흥미로운 토끼굴이기도 합니다!

In [ ]:
# OK, with that, now let's run the model!

outputs = model.generate(inputs, max_new_tokens=80)
outputs[0]

In [ ]:
# Well that doesn't make much sense!
# How about this..

tokenizer.decode(outputs[0])

In [ ]:
# Clean up memory
# Thank you Kuan L. for helping me get this to properly free up memory!
# If you select "Show Resources" on the top right to see GPU memory, it might not drop down right away
# But it does seem that the memory is available for use by new models in the later code.

del model, inputs, tokenizer, outputs
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# Wrapping everything in a function - and adding Streaming and generation prompts

def generate(model, messages, quant=True, max_new_tokens=80):
  tokenizer = AutoTokenizer.from_pretrained(model)
  tokenizer.pad_token = tokenizer.eos_token
  input_ids = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to("cuda")
  attention_mask = torch.ones_like(input_ids, dtype=torch.long, device="cuda")
  streamer = TextStreamer(tokenizer)
  if quant:
    model = AutoModelForCausalLM.from_pretrained(model, quantization_config=quant_config).to("cuda")
  else:
    model = AutoModelForCausalLM.from_pretrained(model).to("cuda")
  outputs = model.generate(input_ids=input_ids, attention_mask=attention_mask, max_new_tokens=max_new_tokens, streamer=streamer)



In [ ]:
messages = [
    {"role": "user", "content": "Tell a light-hearted joke for a room of Data Scientists"}
  ]
generate(GEMMA, messages, quant=False)

In [ ]:
generate(QWEN, messages)

In [ ]:
generate(DEEPSEEK, messages, quant=False, max_new_tokens=500)